# Forest Cover Type Project

# Forest Cover Type Project Exploratory Data Analysis

In the Forest Cover Type competition, we are asked to predict the forest cover type (the predominant kind of tree cover) from cartographic variables. The purpose of this EDA notebook is to provide an overview of how python visualization tools can be used to understand the complex and large dataset. EDA is the first step in this workflow where the decision-making process is initiated for the feature selection. Some valuable insights can be obtained by looking at the distribution of the target, relationship to the target and link between the features.

The study area includes four wilderness areas located in the Roosevelt National Forest of Northern Colorado. Each observation is a 30m x 30m patch. There are seven forest cover types:

1. Spruce/Fir
2. Lodgepole Pine
3. Ponderosa Pine
4. Cottonwood/Willow
5. Aspen
6. Douglas-fir
7. Krummholz

# Data Fields
- **Elevation** elevation in meters
- **Aspect** aspect in degrees azimuth
- **Slope** slope in degrees
- **Horizontal_Distance_To_Hydrology** Horz dist to nearest surface water features
- **Vertical_Distance_To_Hydrology** Vert dist to nearest surface water features
- **Horizontal_Distance_To_Roadways** Horz dist to nearest roadway
- **Hillshade_9am** (0 to 255 index) hillshade index at 9am, summer solstice
- **Hillshade_Noon** (0 to 255 index) hillshade index at noon, summer solstice
- **Hillshade_3pm** (0 to 255 index) hillshade index at 3pm, summer solstice
- **Horizontal_Distance_To_Fire_Points** Horz dist to nearest wildfire ignition points
- **Wilderness_Area** (4 binary cols, 0=abs or 1=pres) wilderness area designation
- **Soil_Type** (40 binary cols, 0=abs or 1=pres) soil type designation
- **Cover_Type** (7 types) forest cover type designation

## Load Data & Setup

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
import plotly.express as px


In [ ]:
dataset = pd.read_csv('../input/forest-cover-type-prediction/train.csv')

# display train data
dataset.head()

In [ ]:
# drop ID column
dataset = dataset.iloc[:,1:]
dataset.head()

In [ ]:
# size of data frame
dataset.shape

Since all variables are numeric integers, there are no need for further conversions.

In [ ]:
# look at the data types of each feature and see if there needs to be any pre-processing
dataset.dtypes

# Exploratory Data Analysis
- Our dataset has **54** features and **1** target variable, `Cover_Type`. 
- From 54 features, 10 are numeric and 44 are categorical.
- From 44 categorical, 40 are `Soil_Type` and 4 of `Wilderness_Area`
- These are the following forest cover types in target variable `Cover_Type`:
    1. Spruce/Fir
    2. Lodgepole Pine
    3. Ponderosa Pine
    4. Cottonwood/Willow
    5. Aspen
    6. Douglas-fir
    7. Krummholz

In [ ]:
# extract all numerical features from train
num_features = dataset.iloc[:,0:10]

# extract all binary features from train
cat_features = dataset.iloc[:, 10:-1]

In [ ]:
num_features.head()

In [ ]:
cat_features.head()

### Part 1. Describe numerical features
- **mean** of the feature varies from 16 to 2749.
- **std** for `Horizontal_Distance_To_Roadways` is the most spread out data, followed by `Horizontal_Distance_To_Fire_Points` and `Elevation`.
- The most desnsed and near to mean is `Slope` followed by all 3 features of `Hillshade`. 
- All features have a minimum value of 0 except `Elevation` and `Vertical_Distance_To_Hydrology` features.
    - `Elevation` has the highest minimum value and `Vertical_Distance_To_Hydrology` has a negative value.
- `Hillshades` features except `Hillshade_3pm` have a similar maximum value.
- `Horizontal_Distance_To_Fire_Points` has the highest maximum value followed by `Horizontal_Distance_To_Roadways` features. They also have the highest ranges of all features.
- `Slope` has the lowest maximum value and range. The `Aspect` feature follows closely behind this same concept.

It is good to note that the reason some features are widely spread and have high values, is because 5 out of the 10 variables are measured in meters. These variables are: `Elevation`,`Horizontal_Distance_To_Hydrology`,`Vertical_Distance_To_Hydrology`,`Horizontal_Distance_To_Roadways`,`Horizontal_Distance_To_Fire_Points`. This makes sense that these have high values and ranges.


Features like `Aspect` and `Slope` are measured in degrees which means there maximum values can't go above 360. `Hillshade` features can only take on a maximum value of 255.


In [ ]:
num_features.describe()

### Part 2. Describe categorical features
- Categorical variables will either have a value of 0 or 1. The **mean** can tell us useful information.
    - `Wilderness_Area3` followed by `Wilderness_Area4` has the highest mean. This signifies that these variables have the most presence in the data compared to other Wilderness Area. Most of our features will consist of `Wilderness_Area3` and `Wilderness_Area4`.
    - The least amount of observations will be seen from `Wilderness_Area2`, and it also has the least standard deviation among `Wilderness_Area`. 
- One more to notice here is that when we add all the mean of `Wilderness_Area` we get a result 0.999999 which is approximately 1. This may mean all the observations can be from any one Wilderness Area.
- Probability wise, the next observation that we get will have a 42.0% probability take from `Wilderness_Area3`, 30.9% probability take from `Wilderness_Area4` and so on for others. 

By looking at these statistics of two different data types, we can see that there are different spreads and uneven amount of distribution. In this case we will feature scale these so that all the features have similar ranges between 0 and 1. Some algorithms can be sensitive to high values hence giving us inappropriate results while some algorithms are not.

In [ ]:
cat_features.describe()

# Dimensionality Reduction
- Based on some EDA, we have lots of observations and features to train the model but to make it computationaly faster and make data visualization easier we will reduce dimensions.

#### Dimensionality Reduction Inferences:
- To understand how PCA works, we will first apply it on numerical features and compare the accuracy of the model with the numerical features.
- Then we will apply it on categorical features and compare the accuracy of the model with the categorical features.
- Lastly, we will create a subset that contains both numerical and categorical features to evaluate model accuracy.

### Spliting Dataset for Numerical features

In [ ]:
x = dataset.iloc[:,0:10]
y = dataset.iloc[:,-1]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 53)

In [ ]:
x.head()

In [ ]:
y.head()

In [ ]:
x_test.shape

In [ ]:
x_train.shape

In [ ]:
y_train.head()

# Prediction on the basis of Numerical Features

### Training Model without dimentionality reduction

In [ ]:
knn = KNeighborsClassifier()

# we are doing supervised learning. knn.fit() is a learning functions, we are telling the labels to learn

knn.fit(x_train, y_train)

In [ ]:
y_pred = knn.predict(x_test)

### Accuracy of model without dimentionality reduction

In [ ]:
accuracy_score(y_test, y_pred)

### Dimentionality reduction

In [ ]:
scaler = StandardScaler()

In [ ]:
#standardization against mean
#standardization transformation

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
pca =  PCA(n_components = 5)

In [ ]:
x_train_trf = pca.fit_transform(x_train)
x_test_trf = pca.transform(x_test)

In [ ]:
pca.explained_variance_
# Eigen values = lamda

In [ ]:
pca.components_.shape
# Eigen vectors 

In [ ]:
#converts into variance percentage
pca.explained_variance_ratio_

In [ ]:
plt.plot(np.cumsum(pca.explained_variance_ratio_))

#from this plot x-axis has PCAs and the y-axis is percentage of variance

### Training model after dimentionality reduction

In [ ]:
knn = KNeighborsClassifier()

In [ ]:
knn.fit(x_train_trf, y_train)

In [ ]:
y_pred = knn.predict(x_test_trf)

In [ ]:
y_pred

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
y_train_trf = y_train.astype(str)

fig = px.scatter(x=x_train_trf[:,0],
                 y=x_train_trf[:,1],
                 color=y_train_trf,
                 color_discrete_sequence=px.colors.qualitative.G10
                )
fig.show()

In [ ]:
for i in range (1, 11):
    pca =  PCA(n_components =  i)
    x_train_trf = pca.fit_transform(x_train)
    x_test_trf = pca.transform(x_test)
    knn = KNeighborsClassifier()
    knn.fit(x_train_trf, y_train)
    y_pred = knn.predict(x_test_trf)
    print(accuracy_score(y_test, y_pred))

### Observation on Accuracy of Model after PCA
- The accuracy of model before Dimentionality Reduction of numerical features predicted with approx 80% accuracy.
- After PCA, the value reduces to 73% accuracy.
- This indicates that PCA has no effect on improving the model's accuracy. It is best for reducing cost and for visualizations.
- Furthermore, increasing PCAs does imporve accuracy as seen by the output of the loop, however, the accuracy reaches a max of 73% and does not increases after that.

### Spliting Dataset for Numerical and Categorical features

In [ ]:
x = dataset.iloc[:,:-1] # feed features to var X
y = dataset.iloc[:,-1] # feed target variable to y

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 53)

In [ ]:
x.head()

In [ ]:
y.head()

# Prediction on the basis of All Features

### Training Model without dimentionality reduction

In [ ]:
knn = KNeighborsClassifier()

# we are doing supervised learning. knn.fit() is a learning functions, we are telling the labels to learn

knn.fit(x_train, y_train)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
y_pred = knn.predict(x_test)

### Accuracy of model without dimentionality reduction

In [ ]:
accuracy_score(y_test, y_pred)

### Dimentionality reduction

In [ ]:
scaler = StandardScaler()

In [ ]:
#standardization against mean
#standardization transformation

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
pca =  PCA(n_components = 10)

In [ ]:
x_train_trf = pca.fit_transform(x_train)
x_test_trf = pca.transform(x_test)

In [ ]:
pca.explained_variance_
# Eigen values = lamda

In [ ]:
pca.components_.shape
# Eigen vectors 

In [ ]:
#converts into variance percentage
pca.explained_variance_ratio_

In [ ]:
np.cumsum(pca.explained_variance_ratio_)

In [ ]:
plt.plot(np.cumsum(pca.explained_variance_ratio_))

#from this plot x-axis has PCAs and the y-axis is percentage of variance

### Training model after dimentionality reduction

In [ ]:
knn = KNeighborsClassifier()

In [ ]:
knn.fit(x_train_trf, y_train)

In [ ]:
y_pred = knn.predict(x_test_trf)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
y_train_trf = y_train.astype(str)

fig = px.scatter(x=x_train_trf[:,0],
                 y=x_train_trf[:,1],
                 color=y_train_trf,
                 color_discrete_sequence=px.colors.qualitative.G10
                )
fig.show()

In [ ]:
for i in range (1, 54):
    pca =  PCA(n_components =  i)
    x_train_trf = pca.fit_transform(x_train)
    x_test_trf = pca.transform(x_test)
    knn = KNeighborsClassifier()
    knn.fit(x_train_trf, y_train)
    y_pred = knn.predict(x_test_trf)
    print(accuracy_score(y_test, y_pred))

### Observation on Accuracy of Model after PCA
- The accuracy of model before Dimentionality Reduction of numerical and categorical features predicted with approx 80% accuracy.
- After PCA, the value reduces to 74% accuracy on 10 PCAs.
- This indicates that PCA has no effect on improving the model's accuracy. It is best for reducing cost and for visualizations.
- Furthermore, increasing PCAs does imporve accuracy as seen by the output of the loop, however, the accuracy reaches a max of 78% which is still lesser than 80% that was observed with PCA.